
### Structured output
Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.

### Pydantic
Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [1]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

# 1. Load environment variables (Make sure GEMINI_API_KEY is in your .env)
load_dotenv()

# 2. Initialize the Gemini 2.5 Flash model
model = init_chat_model("gemini-2.5-flash", model_provider="google_genai")

# 3. Invoke the model
response = model.invoke("write me a 200 words paragraph on Artificial Intelligence")
print(response.content)

Artificial Intelligence (AI) represents the simulation of human intelligence processes by machines, especially computer systems. At its core, AI encompasses capabilities such as learning, reasoning, problem-solving, perception, and language understanding. Far from being a futuristic concept, AI is deeply embedded in our daily lives, silently powering everything from the personalized recommendations on streaming services and e-commerce sites to the intricate algorithms behind search engines and social media feeds. Its transformative impact is felt across virtually every industry, revolutionizing healthcare through advanced diagnostic tools, enhancing transportation with autonomous vehicles, and fortifying finance with sophisticated fraud detection systems. AI systems are designed to analyze massive datasets, identify complex patterns, and make decisions with remarkable speed and accuracy, often surpassing human capabilities in specific, data-intensive tasks. The ongoing advancements in 

In [2]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="This year the movie was released")
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="The movies rating out of 10")

In [3]:
model_with_structure=model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGoogleGenerativeAI(metadata={'lc_versions': {'langchain-core': '1.4.7', 'langchain': '1.3.9'}}, output_version=None, profile={'name': 'Gemini 2.5 Flash', 'release_date': '2025-03-20', 'last_updated': '2025-06-05', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True}, google_api_key=SecretStr('**********'), location=None, model='gemini-2.5-flash', client=<google.genai.client.Client object at 0x0000029774891D30>, default_metadata=(), model_kwargs={}), kwargs={'response_mime_type': 'application/json', 'response_json_schema': {'properties': {'title': 

In [4]:
model.invoke("Provide details about the moview Inception")

AIMessage(content='*Inception* is a critically acclaimed 2010 science fiction heist thriller written, directed, and produced by Christopher Nolan. Known for its complex narrative, stunning visuals, and philosophical themes, it quickly became a modern classic.\n\nHere are the key details about the movie:\n\n---\n\n### **Overview**\n\n*   **Director:** Christopher Nolan\n*   **Writer:** Christopher Nolan\n*   **Producer:** Christopher Nolan, Emma Thomas\n*   **Starring:** Leonardo DiCaprio, Ken Watanabe, Joseph Gordon-Levitt, Marion Cotillard, Elliot Page (credited as Ellen Page), Tom Hardy, Cillian Murphy, Dileep Rao, Tom Berenger, Michael Caine\n*   **Genre:** Science Fiction, Heist, Thriller, Action\n*   **Release Date:** July 16, 2010\n*   **Runtime:** 148 minutes\n*   **Budget:** $160 million\n*   **Box Office:** Over $836 million worldwide\n*   **Awards:** Won 4 Academy Awards (Best Cinematography, Best Sound Editing, Best Sound Mixing, Best Visual Effects) and was nominated for 4 

In [5]:
response=model_with_structure.invoke("Provide details about the moview Inception")
response

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

### Message output alongside parsed structure

In [6]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A movie with details."""
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The year the movie was released")
    director: str = Field(..., description="The director of the movie")
    rating: float = Field(..., description="The movie's rating out of 10")

model_with_structure = model.with_structured_output(Movie, include_raw=True)  

response = model_with_structure.invoke("Provide details about the movie Inception")
response

{'raw': AIMessage(content='{"title":"Inception","year":2010,"director":"Christopher Nolan","rating":8.8}', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019ee43b-fdfe-77d2-9cbb-5758085864ab-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 123, 'total_tokens': 131, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 99}}),
 'parsed': Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8),
 'parsing_error': None}

### Nested Structure

In [7]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie Inception")
response

MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Elliot Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames'), Actor(name='Ken Watanabe', role='Saito')], genres=['Science Fiction', 'Action', 'Thriller'], budget=160.0)

### TypedDict
TypedDict provides a simpler alternative using Python’s built-in typing, ideal when you don’t need runtime validation.

In [8]:
from typing_extensions import TypedDict,Annotated

class MovieDict(TypedDict):
    """A movie with details."""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]


model_withtypedict=model.with_structured_output(MovieDict)
response=model_withtypedict.invoke("Please provide the details of the movie avengers")
response

{'title': 'The Avengers',
 'year': 2012,
 'director': 'Joss Whedon',
 'rating': 8.0}

In [9]:
class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie Avengers")
response

{'title': 'The Avengers',
 'year': 2012,
 'cast': [{'name': 'Robert Downey Jr.', 'role': 'Iron Man'},
  {'name': 'Chris Evans', 'role': 'Captain America'},
  {'name': 'Mark Ruffalo', 'role': 'Hulk'},
  {'name': 'Chris Hemsworth', 'role': 'Thor'},
  {'name': 'Scarlett Johansson', 'role': 'Black Widow'},
  {'name': 'Jeremy Renner', 'role': 'Hawkeye'},
  {'name': 'Samuel L. Jackson', 'role': 'Nick Fury'}],
 'genres': ['Action', 'Sci-Fi', 'Superhero'],
 'budget': 220000000}

In [10]:
model.profile

{'name': 'Gemini 2.5 Flash',
 'release_date': '2025-03-20',
 'last_updated': '2025-06-05',
 'open_weights': False,
 'max_input_tokens': 1048576,
 'max_output_tokens': 65536,
 'text_inputs': True,
 'image_inputs': True,
 'audio_inputs': True,
 'pdf_inputs': True,
 'video_inputs': True,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'structured_output': True,
 'attachment': True,
 'temperature': True,
 'image_url_inputs': True,
 'image_tool_message': True,
 'tool_choice': True}

### DataClasses
A data class is a class typically containing mainly data, although there aren’t really any restrictions. You create it using the @dataclass decorator

In [12]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

# 1. Load environment variables (Make sure GEMINI_API_KEY is in your .env)
load_dotenv()

# 2. Initialize the Gemini 2.5 Flash model
model = init_chat_model("gemini-2.5-flash", model_provider="google_genai")

# 3. Invoke the model
response = model.invoke("write me a 200 words paragraph on Artificial Intelligence")
print(response.content)

Artificial Intelligence (AI) represents the simulation of human intelligence processes by machines, especially computer systems. This broad field encompasses machine learning, deep learning, natural language processing, and computer vision, enabling systems to perform tasks that typically require human intellect. AI-powered systems can learn from vast amounts of data, recognize intricate patterns, make informed decisions, solve complex problems, and even understand and generate human language with remarkable fluency. Their core strength lies in automating cognitive tasks and deriving insights that would be impossible for humans alone.

Today, AI is no longer a futuristic concept but a pervasive technology transforming nearly every sector imaginable. From powering personalized recommendation engines on streaming platforms and optimizing logistics in global supply chains to assisting medical professionals in diagnosing diseases with greater accuracy and enabling autonomous vehicles, its 

In [15]:
import os
from pydantic import BaseModel, Field
from dotenv import load_dotenv
from langchain.agents import create_agent

load_dotenv()

class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

# Use the "provider:model" syntax here:
agent = create_agent(
    model="google_genai:gemini-2.5-flash", 
    response_format=ContactInfo
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

print(result)

{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='1d27f7f3-c2d5-4981-b649-aac7d2b5babd'), AIMessage(content='{"name":"John Doe","email":"john@example.com","phone":"(555) 123-4567"}', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019ee43f-067b-7743-b458-cb6e3ad6ac4f-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 29, 'output_tokens': 587, 'total_tokens': 616, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 556}})], 'structured_response': ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')}


In [16]:
result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

In [17]:
import os
from typing_extensions import TypedDict
from dotenv import load_dotenv
from langchain.agents import create_agent

# 1. Load your environment variables for GEMINI_API_KEY
load_dotenv()

# 2. Define schema using TypedDict
class ContactInfo(TypedDict):
    """Contact information for a person."""
    name: str   # The name of the person
    email: str  # The email address of the person
    phone: str  # The phone number of the person

# 3. Create agent pointing to Gemini
agent = create_agent(
    model="google_genai:gemini-2.5-flash",  # Switched to Gemini
    response_format=ContactInfo
)

# 4. Invoke the agent
result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

# 5. Extract your structured dictionary
print(result["structured_response"])
# Output: {'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}

{'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}


In [18]:
import os
from pydantic.dataclasses import dataclass # Use Pydantic's dataclass variant
from dotenv import load_dotenv
from langchain.agents import create_agent

# 1. Load environment variables
load_dotenv()

# 2. Define schema using Pydantic's dataclass
@dataclass
class ContactInfo:
    """Contact information for a person."""
    name: str   # The name of the person
    email: str  # The email address of the person
    phone: str  # The phone number of the person

# 3. Create agent pointing to Gemini
agent = create_agent(
    model="google_genai:gemini-2.5-flash",
    response_format=ContactInfo
)

# 4. Invoke the agent
result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

# 5. Extract your structured data
print(result["structured_response"])
# Output will be an instance of your ContactInfo dataclass:
# ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')
